In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.models as models

In [ ]:
class SpatialBranch(nn.Module):
    def __init__(self, pretrained=True):
        super().__init__()
        
        resnet = models.resnet18(pretrained=pretrained)
        self.feature_extractor = nn.Sequential(*list(resnet.children())[:-1])
        self.output_dim = 512
        
    def forward(self, x):
        # x: (B*T, 3, H, W)
        x = self.feature_extractor(x)
        x = x.view(x.size(0), -1)
        return x  # (B*T, 512)

In [ ]:
def compute_fft(frames):
    """
    frames: (B, T, 3, H, W)
    Returns: (B, T, 1, H, W)
    """
    
    # Convert to grayscale
    gray = frames.mean(dim=2)  # (B, T, H, W)
    
    # FFT
    fft = torch.fft.fft2(gray)
    fft = torch.abs(fft)
    
    # Log scaling
    fft = torch.log(fft + 1e-8)
    
    return fft.unsqueeze(2)  # add channel dim

In [ ]:
class FrequencyBranch(nn.Module):
    def __init__(self):
        super().__init__()
        
        self.conv = nn.Sequential(
            nn.Conv2d(1, 32, 3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),
            
            nn.Conv2d(32, 64, 3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d(1)
        )
        
        self.output_dim = 64
        
    def forward(self, x):
        x = self.conv(x)
        x = x.view(x.size(0), -1)
        return x  # (B*T, 64)

In [ ]:
class TemporalAttention(nn.Module):
    def __init__(self, hidden_dim):
        super().__init__()
        self.attn = nn.Linear(hidden_dim, 1)
        
    def forward(self, x):
        # x: (B, T, H)
        weights = torch.softmax(self.attn(x), dim=1)
        weighted = x * weights
        return weighted.sum(dim=1)

In [ ]:
class TemporalBranch(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        
        self.lstm = nn.LSTM(
            input_dim,
            256,
            batch_first=True,
            bidirectional=True
        )
        
        self.attention = TemporalAttention(512)
        self.output_dim = 512
        
    def forward(self, x):
        # x: (B, T, input_dim)
        out, _ = self.lstm(x)
        out = self.attention(out)
        return out  # (B, 512)

In [ ]:
class StrongFusionModel(nn.Module):
    def __init__(self):
        super().__init__()
        
        self.spatial = SpatialBranch(pretrained=False)  # set True if internet available
        self.frequency = FrequencyBranch()
        
        fusion_dim = self.spatial.output_dim + self.frequency.output_dim
        
        self.temporal = TemporalBranch(fusion_dim)
        
        self.classifier = nn.Sequential(
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, 1)
        )
        
    def forward(self, frames):
        """
        frames: (B, T, 3, H, W)
        """
        
        B, T, C, H, W = frames.shape
        
        # Compute FFT internally
        fft = compute_fft(frames)
        
        # Flatten time
        frames = frames.view(B*T, C, H, W)
        fft = fft.view(B*T, 1, H, W)
        
        spatial_feat = self.spatial(frames)
        freq_feat = self.frequency(fft)
        
        fused = torch.cat([spatial_feat, freq_feat], dim=1)
        fused = fused.view(B, T, -1)
        
        temporal_feat = self.temporal(fused)
        
        output = self.classifier(temporal_feat)
        
        return output.squeeze(1)

Demo Run

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = StrongFusionModel().to(device)

B = 2
T = 8
H = 128
W = 128

frames = torch.randn(B, T, 3, H, W).to(device)
labels = torch.randint(0, 2, (B,)).float().to(device)

print("Model ready ✅")

In [ ]:
outputs = model(frames)
print("Output shape:", outputs.shape)

In [ ]:
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

model.train()

for step in range(3):
    optimizer.zero_grad()
    
    outputs = model(frames)
    loss = criterion(outputs, labels)
    
    loss.backward()
    optimizer.step()
    
    print(f"Step {step+1} | Loss: {loss.item():.4f}")